# 04 - Train and evaluate

Three things happen here, in order of importance:

1. **The ablation.** gene identity + VAF alone, versus that plus ESM2. If ESM2
   adds nothing, you have a lookup table with extra steps.
2. **Screening metrics.** Accuracy is useless when positives are rare. Report
   sensitivity at 98% specificity and AUPRC.
3. **Cross-validation, not one split.** With a few hundred patients, a single
   train/test split has a +/-0.08 error bar you cannot see.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection")
    # Run every session, not only on first mount: Colab wipes packages on
    # restart, and misspelled package names fail silently without check=True.
    subprocess.run("pip install -q transformers accelerate lightgbm".split(), check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

CFG = yaml.safe_load(open("configs/config.yaml"))
CFG

In [ ]:
from src.model import (build_features, make_model, cross_validate, ablation,
                       evaluate, save_model)

labelled = pd.read_parquet(f'{CFG["data"]["processed_dir"]}/patients_labelled.parquet')
y = labelled["label"].to_numpy()

emb_path = f'{CFG["data"]["processed_dir"]}/X_emb.npy'
X_emb = np.load(emb_path) if os.path.exists(emb_path) else None
print("labels:", np.bincount(y), "| embeddings:",
      None if X_emb is None else X_emb.shape)

In [ ]:
# Prefer the shared panel from notebook 01 over every gene seen in the data.
panel_path = f'{CFG["data"]["processed_dir"]}/shared_panel.csv'
if os.path.exists(panel_path):
    genes = pd.read_csv(panel_path)["gene"].tolist()
    print(f"using shared panel from notebook 01: {len(genes)} genes")
else:
    genes = sorted({g for gs in labelled["genes"] for g in gs})
    print(f"no shared panel found, using all {len(genes)} genes")

X_base, names_base, _ = build_features(labelled, genes, embeddings=None)
X_full, names_full, N_EMB = (
    build_features(labelled, genes, embeddings=X_emb, n_pca=CFG["model"]["n_pca"])
    if X_emb is not None else (X_base, names_base, 0))
print("raw embedding columns:", N_EMB)

In [ ]:
kind = CFG["model"]["kind"]
cv_base = cross_validate(X_base, y, kind, n_splits=CFG["model"]["n_splits"],
                         target_spec=CFG["model"]["target_specificity"])
cv_base.round(3)

In [ ]:
if X_emb is not None:
    tbl = ablation(X_base, X_full, y, kind, n_emb=N_EMB,
                   n_pca=CFG["model"]["n_pca"],
                   n_splits=CFG["model"]["n_splits"],
                   target_spec=CFG["model"]["target_specificity"])
    print(tbl.round(3))
else:
    print("No embeddings yet - run notebook 03 to make this comparison.")

**How to read the ablation.** A gain of +0.01 AUC that is smaller than the
fold-to-fold standard deviation is noise, not a result.

In [ ]:
print(f"fold-to-fold AUC std (baseline): {cv_base['auc'].std():.3f}")
print("Any BioNeMo ESM-2 gain smaller than this is not yet evidence of anything.")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, roc_auc_score

def oof_scores(X, y, kind, n_splits=5, seed=0, n_emb=0):
    p = np.zeros(len(y))
    for tr, te in StratifiedKFold(n_splits, shuffle=True,
                                  random_state=seed).split(X, y):
        m = make_model(kind, n_emb=n_emb, n_pca=CFG["model"]["n_pca"])
        m.fit(X[tr], y[tr])
        p[te] = m.predict_proba(X[te])[:, 1]
    return p

plt.figure(figsize=(6, 5))
curves = [("gene + VAF", X_base)] + ([("+ BioNeMo ESM-2", X_full)] if X_emb is not None else [])
for label, X in curves:
    p = oof_scores(X, y, kind, n_emb=N_EMB if label != "gene + VAF" else 0)
    fpr, tpr, _ = roc_curve(y, p)
    plt.plot(fpr, tpr, label=f"{label} (AUC {roc_auc_score(y, p):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=.8)
plt.axvline(1 - CFG["model"]["target_specificity"], color="r", ls=":",
            label=f'{CFG["model"]["target_specificity"]:.0%} specificity')
plt.xlabel("1 - specificity"); plt.ylabel("sensitivity")
plt.legend(); plt.title("Out-of-fold ROC"); plt.show()

In [ ]:
# Which features carry the model? Read this before believing the AUC.
X_use, names_use = (X_full, names_full) if X_emb is not None else (X_base, names_base)
EMB_USE = N_EMB if X_emb is not None else 0

m = make_model("logreg", n_emb=EMB_USE, n_pca=CFG["model"]["n_pca"])
m.fit(X_use, y)

# After EmbeddingPCA the columns are: named features + esm_pc0..k
n_named = len(names_use) - EMB_USE
cols = names_use[:n_named] + [f"esm_pc{i}" for i in
                              range(m.named_steps["clf"].coef_.shape[1] - n_named)]
coef = pd.Series(m.named_steps["clf"].coef_[0], index=cols)
print(coef.reindex(coef.abs().sort_values(ascending=False).index).head(20).round(3))


In [ ]:
final = make_model(kind, n_emb=EMB_USE, n_pca=CFG["model"]["n_pca"])
final.fit(X_use, y)

p_oof = oof_scores(X_use, y, kind, n_emb=EMB_USE)
metrics = evaluate(y, p_oof, CFG["model"]["target_specificity"])
print(metrics)

# The pipeline carries its own fitted PCA, so joblib saves everything needed.
save_model(final, CFG["model"]["out_path"], meta={
    "genes": genes,
    "kind": kind,
    "uses_embeddings": X_emb is not None,
    "n_emb": int(EMB_USE),
    "threshold": metrics["threshold"],
    "label_strategy": CFG["labels"]["strategy"],
    "oof_metrics": {k: float(v) for k, v in metrics.items()},
})
